In [67]:
!pip install scikit-learn

In [68]:
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB

In [69]:
from collections import Counter

import numpy as np
from nltk.util import ngrams

## Old Testament

In [88]:
# prepare Peshitta OT from text-fabric
from tf.app import use

api_handler = use("etcbc/peshitta", hoist=globals(), version="0.2")

**Locating corpus resources ...**

Name,# of nodes,# slots / node,% coverage
book,65,6566.69,100
chapter,1269,336.36,100
verse,31341,13.62,100
word,426835,1.00,100


In [89]:
target_books = {
    "Genesis": list(range(1, 51)),
    "Exodus": list(range(1, 22)),
    "Deuteronomy": list(range(1, 21)),
}

In [90]:
# Parse the dataset and get verses
ot_verses = []  # each verse in the format of "(`verse reference`: str, `transliteration as a list of words`: list[str])"
ot_book_ids = []

for book in Fs("book@en").items():
    if book[1] in target_books:
        print(book[1])
        chapters = L.d(book[0], otype="chapter")
        # results = "Verse Reference,Lemmatised Transliteration,Plain Transliteration,Original Syriac Text\n"
        num_book_verses = 0
        for chapter in chapters:
            if int(F.chapter.v(chapter)) in target_books[book[1]]:
                for verse in L.d(chapter, otype="verse"):
                    verse_ref = f"{book[1]}: Chapter {F.chapter.v(chapter)} Verse {F.verse.v(verse)}"
                    # get all words in this verse
                    words = L.d(verse, otype="word")
                    # transliteration of this verse as a list of words
                    translit_verse = [F.word_etcbc.v(w_id) for w_id in words]
                    ot_verses.append((verse_ref, translit_verse))
                    num_book_verses += 1
        ot_book_ids.append((book[1], num_book_verses))
print(ot_book_ids)

Genesis
Exodus
Deuteronomy
[('Genesis', 1533), ('Exodus', 582), ('Deuteronomy', 555)]


In [91]:
# First Vocab construction
ot_n_gram_counters = []
ot_chargram_counters = []

n_gram_vocabs = None
chargram_vocabs = None

word_cnt = 0

span = 3  # n of n-gram

for verse in ot_verses:
    # Format the verse to feed into ngrams()
    word_cnt += len(verse[1])
    target_joined = " ".join(verse[1])
    # Generate and count word- and character-ngrams
    n_grams = list(ngrams(verse[1], span))
    chargrams = list(ngrams(target_joined, span))
    n_gram_count = Counter(n_grams)
    chargram_count = Counter(chargrams)
    # Keep the ngram counters
    ot_n_gram_counters.append(n_gram_count)
    ot_chargram_counters.append(chargram_count)

    if n_gram_vocabs is None:
        n_gram_vocabs = n_gram_count.copy()
        chargram_vocabs = chargram_count.copy()
    else:
        n_gram_vocabs.update(n_gram_count)
        chargram_vocabs.update(chargram_count)

print(f"Total word count: {word_cnt}")

print(len(n_gram_vocabs.keys()))

print(len(chargram_vocabs.keys()))

Total word count: 36852
29081
6744


## New Testament

In [92]:
# Prepare Peshitta NT from text-fabric

api_handler = use("etcbc/syrnt", hoist=globals(), version="0.1")

**Locating corpus resources ...**

Name,# of nodes,# slots / node,% coverage
book,27,4060.74,100
chapter,260,421.69,100
lexeme,3038,36.09,100
verse,7957,13.78,100
word,109640,1.00,100


In [93]:
Fs("book@en").items()

dict_items([(109641, 'Matthew'), (109642, 'Mark'), (109643, 'Luke'), (109644, 'John'), (109645, 'Acts'), (109646, 'Romans'), (109647, '1_Corinthians'), (109648, '2_Corinthians'), (109649, 'Galatians'), (109650, 'Ephesians'), (109651, 'Philippians'), (109652, 'Colossians'), (109653, '1_Thessalonians'), (109654, '2_Thessalonians'), (109655, '1_Timothy'), (109656, '2_Timothy'), (109657, 'Titus'), (109658, 'Philemon'), (109659, 'Hebrews'), (109660, 'James'), (109661, '1_Peter'), (109662, '2_Peter'), (109663, '1_John'), (109664, '2_John'), (109665, '3_John'), (109666, 'Jude'), (109667, 'Revelation')])

In [94]:
target_books = {
    "Matthew": list(range(1, 31)),
    "Mark": list(range(1, 17)),
    "Luke": list(range(1, 24)),
    "John": list(range(1, 22)),
    "Acts": list(range(1, 29)),
}

In [95]:
# Parse the dataset and get verses
nt_verses = []  # each verse in the format of "(`verse reference`: str, `transliteration as a list of words`: list[str])"
nt_book_ids = []

for book in Fs("book@en").items():
    if book[1] in target_books.keys():
        print(book[1])
        chapters = L.d(book[0], otype="chapter")
        # results = "Verse Reference,Lemmatised Transliteration,Plain Transliteration,Original Syriac Text\n"
        verse_counts = 0

        for chapter in chapters:
            if int(F.chapter.v(chapter)) in target_books[book[1]]:
                for verse in L.d(chapter, otype="verse"):
                    verse_ref = f"{book[1]}: Chapter {F.chapter.v(chapter)} Verse {F.verse.v(verse)}"
                    # get all words in this verse
                    words = L.d(verse, otype="word")
                    # transliteration of this verse as a list of words
                    translit_verse = [F.word_etcbc.v(w_id) for w_id in words]
                    nt_verses.append((verse_ref, translit_verse))
                    verse_counts += 1
        nt_book_ids.append((book[1], verse_counts))

Matthew
Mark
Luke
John
Acts


In [96]:
# Extend the OT Vocab with NT
nt_n_gram_counters = []
nt_chargram_counters = []

word_cnt = 0

span = 3  # n of n-gram

for verse in nt_verses:
    word_cnt += len(verse[1])
    target_joined = " ".join(verse[1])
    n_grams = list(ngrams(verse[1], span))
    chargrams = list(ngrams(target_joined, span))
    n_gram_count = Counter(n_grams)
    chargram_count = Counter(chargrams)
    nt_n_gram_counters.append(n_gram_count)
    nt_chargram_counters.append(chargram_count)

    if n_gram_vocabs is None:
        n_gram_vocabs = n_gram_count.copy()
        chargram_vocabs = chargram_count.copy()
    else:
        n_gram_vocabs.update(n_gram_count)
        chargram_vocabs.update(chargram_count)

print(f"Total word count: {word_cnt}")

print(len(n_gram_vocabs.keys()))

print(len(chargram_vocabs.keys()))

Total word count: 65168
77852
7875


## Vectorise the verses using the constructed vocabulary

### Jewish

In [97]:
# Count the number of each n-gram per verse
# and vectorise by the BoW approach

assert len(ot_verses) == len(ot_n_gram_counters)
assert len(ot_verses) == len(ot_chargram_counters)

ot_n_gram_feat = []
ot_chargram_feat = []

for i in range(len(ot_verses)):
    n_gram_bow = dict.fromkeys(n_gram_vocabs.keys(), 0)
    n_gram_bow.update(ot_n_gram_counters[i])
    features = list(n_gram_bow.values())
    ot_n_gram_feat.append(features)

    chargram_bow = dict.fromkeys(chargram_vocabs.keys(), 0)
    chargram_bow.update(ot_chargram_counters[i])
    ot_chargram_feat.append(list(chargram_bow.values()))

### Christian

In [98]:
# Prepare feature arrays for "Christian" category
# Count the number of each n-gram per verse
# and vectorise in BoW approach

assert len(nt_verses) == len(nt_n_gram_counters)
assert len(nt_verses) == len(nt_chargram_counters)

nt_n_gram_feat = []
nt_chargram_feat = []

for i in range(len(nt_verses)):
    n_gram_bow = dict.fromkeys(n_gram_vocabs.keys(), 0)
    n_gram_bow.update(nt_n_gram_counters[i])
    nt_n_gram_feat.append(list(n_gram_bow.values()))

    chargram_bow = dict.fromkeys(chargram_vocabs.keys(), 0)
    chargram_bow.update(nt_chargram_counters[i])
    nt_chargram_feat.append(list(chargram_bow.values()))

## Label the data

In [99]:
# Merge the feature arrays
verse_n_gram_feat = []
verse_chargram_feat = []
verse_labels = []  # 0 for "Jewish" and 1 for "Christian" verse

verse_n_gram_feat.extend(ot_n_gram_feat)
verse_n_gram_feat.extend(nt_n_gram_feat)
verse_chargram_feat.extend(ot_chargram_feat)
verse_chargram_feat.extend(nt_chargram_feat)

# Create the label array
verse_labels.extend([0 for i in range(len(ot_verses))])
verse_labels.extend([1 for i in range(len(nt_verses))])

print(verse_labels[0:10])
divide = len(ot_verses) - 1
print(verse_labels[divide : divide + 10])

[0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 1, 1, 1, 1, 1, 1, 1, 1, 1]


In [100]:
# Convert the arrays to numpy array for utility
n_gram_feats = np.array(verse_n_gram_feat)
chargram_feats = np.array(verse_chargram_feat)
labels = np.array(verse_labels)

In [83]:
np.save("./out/verse_n_gram_array.npy", verse_n_gram_feat)
np.save("./out/chargram_array.npy", verse_chargram_feat)
np.save("./out/labels.npy", labels)

In [101]:
print(ot_book_ids)
print(nt_book_ids)

[('Genesis', 1533), ('Exodus', 582), ('Deuteronomy', 555)]
[('Matthew', 1071), ('Mark', 678), ('Luke', 1098), ('John', 879), ('Acts', 1007)]


In [102]:
import pickle
from pathlib import Path

with Path("./out/ot_verse_nums.pickle").open(mode="wb") as f:
    pickle.dump(ot_book_ids, f)

with Path("./out/nt_verse_nums.pickle").open(mode="wb") as f:
    pickle.dump(nt_book_ids, f)

In [114]:
nt_book_ids

[('Matthew', 1071),
 ('Mark', 678),
 ('Luke', 1098),
 ('John', 879),
 ('Acts', 1007)]

In [167]:
import json

with Path("./out/ot_verses.json").open(mode="w") as fp:
    json.dump(ot_verses, fp)
len(ot_verses)

2670

In [166]:
ot_prev_verses = sum([book[1] for book in ot_book_ids[:0]])
ot_verses[ot_prev_verses + 0]

('Genesis: Chapter 1 Verse 1',
 ['BRCJT', 'BR>', '>LH>', 'JT', 'CMJ>', 'WJT', '>R<>'])

In [168]:
with Path("./out/nt_verses.json").open(mode="w") as fp:
    json.dump(nt_verses, fp)
len(nt_verses)

4733

In [165]:
nt_prev_verses = sum([book[1] for book in nt_book_ids[:4]])
nt_verses[nt_prev_verses + 745]

('Acts: Chapter 21 Verse 5',
 ['WMN',
  'BTR',
  'HLJN',
  'JWMT>',
  'NPQN',
  'DN>ZL',
  'B>WRX>',
  'WMLWJN',
  'HWW',
  'LN',
  'KLHWN',
  'HNWN',
  'WNCJHWN',
  'WBNJHWN',
  '<DM>',
  'LBR',
  'MN',
  'MDJNT>',
  'WQ<DW',
  '<L',
  'BWRKJHWN',
  '<L',
  'JD',
  'JM>',
  'WYLJW'])

## Training the algorithm

In [42]:
n_gram_train, n_gram_test, n_gram_train_y, n_gram_test_y = train_test_split(
    n_gram_feats, labels, test_size=0.2, random_state=0
)

gnb = GaussianNB()

y_pred = gnb.fit(n_gram_train, n_gram_train_y).predict(n_gram_test)

print(
    "Number of mislabeled points out of a total %d points : %d"
    % (n_gram_test.shape[0], (n_gram_test_y != y_pred).sum())
)

Number of mislabeled points out of a total 1481 points : 406


In [43]:
chargram_X_train, chargram_X_test, chargram_y_train, chargram_y_test = (
    train_test_split(chargram_feats, labels, test_size=0.2, random_state=0)
)

gnb = GaussianNB()

y_pred = gnb.fit(chargram_X_train, chargram_y_train).predict(chargram_X_test)

print(
    "Number of mislabeled points out of a total %d points : %d"
    % (chargram_X_test.shape[0], (chargram_y_test != y_pred).sum())
)

Number of mislabeled points out of a total 1481 points : 57


### Quick evaluations

In [44]:
from sklearn.model_selection import cross_val_score

In [45]:
fold = 5

print(f"Gaussian Naïve Bayes with Word n-grams, {fold}-fold cross-validation:")

# Calculate Accuracy through Cross-Validation
cv_scores = cross_val_score(
    gnb, n_gram_train, n_gram_train_y, cv=fold
)  # cross-validation scores
print(
    "%0.6f accuracy with a standard deviation of %0.6f"
    % (cv_scores.mean(), cv_scores.std())
)

# Calculate F1 score through Cross-Validation
cv_fscores = cross_val_score(
    gnb, n_gram_train, n_gram_train_y, cv=fold, scoring="f1_macro"
)
print(
    "%0.6f F1 score with a standard deviation of %0.6f"
    % (cv_fscores.mean(), cv_fscores.std())
)

Gaussian Naïve Bayes with Word n-grams, 5-fold cross-validation:
0.731508 accuracy with a standard deviation of 0.008905
0.731355 F1 score with a standard deviation of 0.008857


In [46]:
fold = 5

print(
    f"Gaussian Naïve Bayes with Character n-grams, {fold}-fold cross-validation:"
)

# Calculate Accuracy through Cross-Validation
cv_scores = cross_val_score(
    gnb, chargram_X_train, chargram_y_train, cv=fold
)  # cross-validation scores
print(
    "%0.6f accuracy with a standard deviation of %0.6f"
    % (cv_scores.mean(), cv_scores.std())
)

# Calculate F1 score through Cross-Validation
cv_fscores = cross_val_score(
    gnb, chargram_X_train, chargram_y_train, cv=fold, scoring="f1_macro"
)
print(
    "%0.6f F1 score with a standard deviation of %0.6f"
    % (cv_fscores.mean(), cv_fscores.std())
)

Gaussian Naïve Bayes with Character n-grams, 5-fold cross-validation:
0.965553 accuracy with a standard deviation of 0.002085
0.962554 F1 score with a standard deviation of 0.002242


In [47]:
y_pred = gnb.fit(chargram_X_test, chargram_y_test).predict(chargram_X_test)
print()

In [48]:
# cv_fscores = cross_val_score(gnb, n_gram_train, n_gram_train_y, cv=fold, scoring='f1_macro')

## SVM

In [49]:
# chargram_X_train, chargram_X_test, chargram_y_train, chargram_y_test = train_test_split(chargram_feats, labels, test_size=0.2, random_state=0)

# gnb = GaussianNB()

# y_pred = gnb.fit(chargram_X_train, chargram_y_train).predict(chargram_X_test)

# fold = 5

# print(f"Support Vector Machine, {fold}-fold cross-validation:")

# # Calculate Accuracy through Cross-Validation
# cv_scores = cross_val_score(gnb, chargram_X_train, chargram_y_train, cv=fold) # cross-validation scores
# print("%0.6f accuracy with a standard deviation of %0.6f" % (cv_scores.mean(), cv_scores.std()))

# # Calculate F1 score through Cross-Validation
# cv_fscores = cross_val_score(gnb, chargram_X_train, chargram_y_train, cv=fold, scoring='f1_macro')
# print("%0.6f F1 score with a standard deviation of %0.6f" % (cv_fscores.mean(), cv_fscores.std()))